# Notebook 3 – RDF graph construction and ontology design

In this notebook, we transform the outputs of the crawling and information extraction stages into an **initial private knowledge base in RDF**.

The goals of this notebook are:
1. Load the movie metadata and extracted information produced in the previous notebooks.
2. Define a small ontology for the movie domain.
3. Construct an RDF graph using normalized URIs, typed entities, and explicit predicates.
4. Save the initial graph and ontology for the next stages of the project.

This notebook corresponds to the **Knowledge Base Construction** stage of the project. The resulting graph will be reused later for:
- entity linking and predicate alignment,
- KB expansion via SPARQL,
- SWRL reasoning,
- knowledge graph embeddings,
- RDF/SPARQL-based RAG.

The main outputs of this notebook will be:
- `kg_artifacts/initial_graph.ttl`
- `kg_artifacts/ontology.ttl`
- basic KB statistics (triples, entities, relations)

> Design note: this notebook focuses on building a **clean initial RDF graph**, not the expanded KB. We keep the ontology simple and practical so that the next notebooks can align and extend it more easily.

In [2]:
# Cell 2 — Install dependencies, imports, and load Notebook 1 + 2 outputs

# Run once in Colab if needed
%pip install -q rdflib pandas

import os
import re
import json
import unicodedata
import pandas as pd

from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

# Paths
films_path = "/content/wikidata_films.json"
cleaned_plots_path = "/content/cleaned_plots.jsonl"
entities_path = "/content/extracted_entities.csv"
relations_path = "/content/extracted_relations.csv"

# Output folder
os.makedirs("/content/kg_artifacts", exist_ok=True)

# Check files
for path in [films_path, cleaned_plots_path, entities_path, relations_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

# Load films metadata
with open(films_path, "r", encoding="utf-8") as f:
    films = json.load(f)

films_df = pd.DataFrame(films)

# Load cleaned plots JSONL
cleaned_plots = []
with open(cleaned_plots_path, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            cleaned_plots.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Skipping malformed JSONL line {line_number}: {e}")

cleaned_plots_df = pd.DataFrame(cleaned_plots)

# Load extracted entities and relations
entities_df = pd.read_csv(entities_path)
relations_df = pd.read_csv(relations_path)

print("=== Notebook 3 input summary ===")
print(f"Films: {len(films_df)}")
print(f"Cleaned summaries: {len(cleaned_plots_df)}")
print(f"Extracted entities: {len(entities_df)}")
print(f"Extracted relations: {len(relations_df)}")

display(films_df.head(2))
display(cleaned_plots_df.head(2))
display(entities_df.head(2))
display(relations_df.head(2))

=== Notebook 3 input summary ===
Films: 998
Cleaned summaries: 580
Extracted entities: 5262
Extracted relations: 655


,uri,id,label,wikipedia_title,date,directors,genres,countries,imdb_ids,cast,awards,production_companies,followed_by,preceded_by
0,http://www.wikidata.org/entity/Q138793787,Q138793787,"1+1+1 Life, Love, Chaos","1+1+1 Life, Love, Chaos",None,[],[],[],[],[],[],[],[],[]
1,http://www.wikidata.org/entity/Q134083298,Q134083298,1-800-On-Her-Own,1-800-On-Her-Own,None,[Dana Flor],[],[United States],[tt32147765],[],[],[],[],[]


,id,title,wikidata_uri,summary,page_url
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,https://en.wikipedia.org/wiki/Ajab_Raat_Ni_Gaj...
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Alanaati Ramchandrudu is a 2024 Indian Telugu-...,https://en.wikipedia.org/wiki/Alanaati_Ramchan...


,film_id,title,wikidata_uri,entity_text,entity_label,sentence,start_char,end_char
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Raat Ni Gajab Vaat,PERSON,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,5,23
1,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,2024,DATE,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,29,33


,film_id,title,wikidata_uri,subject,predicate,object,sentence,extraction_method
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Bhavya Gandhi,star,Patel,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",root_verb_fallback
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Hymavathi Jadapolu,produce,Hyniva Creations LLP,"It is produced by Hymavathi Jadapolu, Sreeram ...",root_verb_fallback


In [3]:
# Cell 3 — Define namespaces, URI helpers, and utility functions

EX = Namespace("http://example.org/movie#")
EXREL = Namespace("http://example.org/movie/relation/")
SCHEMA = Namespace("http://schema.org/")

def slugify(value: str) -> str:
    """
    Convert free text into a URI-safe slug.
    """
    if value is None:
        return "unknown"
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = value.strip("_")
    return value or "unknown"

def ensure_list(value):
    """
    Normalize a field into a Python list.
    """
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [value]

def make_film_uri(film_id: str) -> URIRef:
    """
    Use stable film IDs for film URIs.
    """
    return EX[f"film_{film_id}"]

def make_entity_uri(text: str) -> URIRef:
    """
    Create a URI for extracted or metadata-linked entities.
    """
    return EX[f"entity_{slugify(text)}"]

def make_relation_uri(predicate_text: str) -> URIRef:
    """
    Create a URI for extracted relation predicates.
    """
    return EXREL[slugify(predicate_text)]

def add_type_and_label(graph: Graph, uri: URIRef, class_uri: URIRef = None, label: str = None):
    """
    Add rdf:type and rdfs:label if provided.
    """
    if class_uri is not None:
        graph.add((uri, RDF.type, class_uri))
    if label:
        graph.add((uri, RDFS.label, Literal(str(label))))

def entity_class_from_ner_label(label: str):
    """
    Map spaCy NER labels to ontology classes.
    """
    mapping = {
        "PERSON": EX.Person,
        "ORG": EX.Organization,
        "GPE": EX.Place,
        "DATE": EX.TemporalExpression,
    }
    return mapping.get(label)

# Build a quick lookup for NER-derived entity types (majority label if repeated)
entity_type_lookup = {}
if not entities_df.empty:
    grouped = entities_df.groupby("entity_text")["entity_label"]
    for entity_text, labels in grouped:
        mode_labels = labels.mode()
        entity_type_lookup[entity_text] = mode_labels.iloc[0] if not mode_labels.empty else labels.iloc[0]

print("Namespaces and helpers ready.")

Namespaces and helpers ready.


In [4]:
# Cell 4 — Define the ontology graph

def build_ontology_graph():
    ontology = Graph()

    ontology.bind("ex", EX)
    ontology.bind("exrel", EXREL)
    ontology.bind("schema", SCHEMA)
    ontology.bind("rdf", RDF)
    ontology.bind("rdfs", RDFS)
    ontology.bind("owl", OWL)
    ontology.bind("xsd", XSD)

    ontology_uri = EX["ontology"]
    ontology.add((ontology_uri, RDF.type, OWL.Ontology))
    ontology.add((ontology_uri, RDFS.label, Literal("Movies Knowledge Graph Ontology")))

    # Classes
    classes = {
        EX.Film: "Film",
        EX.Person: "Person",
        EX.Organization: "Organization",
        EX.Place: "Place",
        EX.TemporalExpression: "TemporalExpression",
        EX.Genre: "Genre",
        EX.Award: "Award",
        EX.Company: "Company",
        EX.Country: "Country",
    }

    for class_uri, label in classes.items():
        ontology.add((class_uri, RDF.type, OWL.Class))
        ontology.add((class_uri, RDFS.label, Literal(label)))

    # Subclass relationship
    ontology.add((EX.Country, RDFS.subClassOf, EX.Place))

    # Object properties
    object_properties = {
        EX.directedBy: ("directedBy", EX.Film, EX.Person),
        EX.hasCastMember: ("hasCastMember", EX.Film, EX.Person),
        EX.hasGenre: ("hasGenre", EX.Film, EX.Genre),
        EX.hasCountry: ("hasCountry", EX.Film, EX.Country),
        EX.wonAward: ("wonAward", EX.Film, EX.Award),
        EX.producedBy: ("producedBy", EX.Film, EX.Company),
        EX.followedBy: ("followedBy", EX.Film, EX.Film),
        EX.precededBy: ("precededBy", EX.Film, EX.Film),
        EX.mentionsEntity: ("mentionsEntity", EX.Film, RDFS.Resource),
    }

    for prop_uri, (label, domain, range_) in object_properties.items():
        ontology.add((prop_uri, RDF.type, OWL.ObjectProperty))
        ontology.add((prop_uri, RDFS.label, Literal(label)))
        ontology.add((prop_uri, RDFS.domain, domain))
        ontology.add((prop_uri, RDFS.range, range_))

    # Datatype properties
    datatype_properties = {
        EX.title: ("title", EX.Film, XSD.string),
        EX.releaseDate: ("releaseDate", EX.Film, XSD.date),
        EX.imdbId: ("imdbId", EX.Film, XSD.string),
        EX.sourceSummary: ("sourceSummary", EX.Film, XSD.string),
        EX.pageURL: ("pageURL", EX.Film, XSD.anyURI),
    }

    for prop_uri, (label, domain, dtype) in datatype_properties.items():
        ontology.add((prop_uri, RDF.type, OWL.DatatypeProperty))
        ontology.add((prop_uri, RDFS.label, Literal(label)))
        ontology.add((prop_uri, RDFS.domain, domain))
        ontology.add((prop_uri, RDFS.range, dtype))

    return ontology

ontology_graph = build_ontology_graph()
print(f"Ontology graph created with {len(ontology_graph)} triples.")

Ontology graph created with 77 triples.


In [5]:
# Cell 5 — Build the initial RDF graph from structured film metadata

kg_graph = Graph()
kg_graph.bind("ex", EX)
kg_graph.bind("exrel", EXREL)
kg_graph.bind("schema", SCHEMA)

film_uri_lookup = {}

def add_linked_entities(graph: Graph, subject_uri: URIRef, predicate_uri: URIRef, values, class_uri: URIRef):
    for value in ensure_list(values):
        if not value:
            continue
        entity_uri = make_entity_uri(value)
        add_type_and_label(graph, entity_uri, class_uri, value)
        graph.add((subject_uri, predicate_uri, entity_uri))

for film in films_df.to_dict(orient="records"):
    film_id = film["id"]
    film_uri = make_film_uri(film_id)
    film_uri_lookup[film_id] = film_uri

    add_type_and_label(kg_graph, film_uri, EX.Film, film.get("label"))
    kg_graph.add((film_uri, EX.title, Literal(film.get("label", ""))))

    release_date = film.get("date")
    if release_date:
        kg_graph.add((film_uri, EX.releaseDate, Literal(release_date, datatype=XSD.date)))

    # Structured metadata links
    add_linked_entities(kg_graph, film_uri, EX.directedBy, film.get("directors"), EX.Person)
    add_linked_entities(kg_graph, film_uri, EX.hasGenre, film.get("genres"), EX.Genre)
    add_linked_entities(kg_graph, film_uri, EX.hasCountry, film.get("countries"), EX.Country)
    add_linked_entities(kg_graph, film_uri, EX.hasCastMember, film.get("cast"), EX.Person)
    add_linked_entities(kg_graph, film_uri, EX.wonAward, film.get("awards"), EX.Award)
    add_linked_entities(kg_graph, film_uri, EX.producedBy, film.get("production_companies"), EX.Company)

    for sequel in ensure_list(film.get("followed_by")):
        sequel_uri = make_entity_uri(sequel)
        add_type_and_label(kg_graph, sequel_uri, EX.Film, sequel)
        kg_graph.add((film_uri, EX.followedBy, sequel_uri))

    for prequel in ensure_list(film.get("preceded_by")):
        prequel_uri = make_entity_uri(prequel)
        add_type_and_label(kg_graph, prequel_uri, EX.Film, prequel)
        kg_graph.add((film_uri, EX.precededBy, prequel_uri))

    for imdb_id in ensure_list(film.get("imdb_ids")):
        if imdb_id:
            kg_graph.add((film_uri, EX.imdbId, Literal(imdb_id)))

print(f"Structured metadata graph now has {len(kg_graph)} triples.")

Structured metadata graph now has 17800 triples.


In [6]:
# Cell 6 — Add cleaned summaries and NER-derived entity nodes

for row in cleaned_plots_df.to_dict(orient="records"):
    film_id = row.get("id")
    film_uri = film_uri_lookup.get(film_id, make_film_uri(film_id))

    # Add summaries and page URLs as literals
    summary_text = row.get("summary", "")
    page_url = row.get("page_url", "")

    if summary_text:
        kg_graph.add((film_uri, EX.sourceSummary, Literal(summary_text)))
    if page_url:
        kg_graph.add((film_uri, EX.pageURL, Literal(page_url, datatype=XSD.anyURI)))

# Add entity nodes and mentionsEntity links
for row in entities_df.to_dict(orient="records"):
    film_id = row.get("film_id")
    film_uri = film_uri_lookup.get(film_id, make_film_uri(film_id))

    entity_text = row.get("entity_text")
    entity_label = row.get("entity_label")
    entity_uri = make_entity_uri(entity_text)

    entity_class = entity_class_from_ner_label(entity_label)
    add_type_and_label(kg_graph, entity_uri, entity_class, entity_text)

    kg_graph.add((film_uri, EX.mentionsEntity, entity_uri))

print(f"Graph after adding summaries and extracted entities: {len(kg_graph)} triples.")

Graph after adding summaries and extracted entities: 30715 triples.


In [7]:
# Cell 7 — Add extracted candidate relations from Notebook 2

dynamic_predicates = set()

for row in relations_df.to_dict(orient="records"):
    subject_text = str(row.get("subject", "")).strip()
    predicate_text = str(row.get("predicate", "")).strip()
    object_text = str(row.get("object", "")).strip()

    if not subject_text or not predicate_text or not object_text:
        continue

    subject_uri = make_entity_uri(subject_text)
    object_uri = make_entity_uri(object_text)
    predicate_uri = make_relation_uri(predicate_text)

    dynamic_predicates.add((predicate_uri, predicate_text))

    # Try to type subject/object from entity lookup if we have it
    subj_label = entity_type_lookup.get(subject_text)
    obj_label = entity_type_lookup.get(object_text)

    subj_class = entity_class_from_ner_label(subj_label) if subj_label else None
    obj_class = entity_class_from_ner_label(obj_label) if obj_label else None

    add_type_and_label(kg_graph, subject_uri, subj_class, subject_text)
    add_type_and_label(kg_graph, object_uri, obj_class, object_text)

    kg_graph.add((predicate_uri, RDF.type, RDF.Property))
    kg_graph.add((predicate_uri, RDFS.label, Literal(predicate_text)))

    kg_graph.add((subject_uri, predicate_uri, object_uri))

print(f"Graph after adding extracted relations: {len(kg_graph)} triples.")
print(f"Dynamic extracted predicates added: {len(dynamic_predicates)}")

Graph after adding extracted relations: 31562 triples.
Dynamic extracted predicates added: 96


In [8]:
# Cell 8 — Save ontology, data graph, and combined graph

ontology_path = "/content/kg_artifacts/ontology.ttl"
initial_graph_ttl_path = "/content/kg_artifacts/initial_graph.ttl"
initial_graph_nt_path = "/content/kg_artifacts/initial_graph.nt"
combined_graph_path = "/content/kg_artifacts/combined_graph.ttl"

combined_graph = ontology_graph + kg_graph

ontology_graph.serialize(destination=ontology_path, format="turtle")
kg_graph.serialize(destination=initial_graph_ttl_path, format="turtle")
kg_graph.serialize(destination=initial_graph_nt_path, format="nt")
combined_graph.serialize(destination=combined_graph_path, format="turtle")

print("Saved ontology to:", ontology_path)
print("Saved initial RDF graph (TTL) to:", initial_graph_ttl_path)
print("Saved initial RDF graph (NT) to:", initial_graph_nt_path)
print("Saved combined ontology+graph to:", combined_graph_path)

/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Saved ontology to: /content/kg_artifacts/ontology.ttl
Saved initial RDF graph (TTL) to: /content/kg_artifacts/initial_graph.ttl
Saved initial RDF graph (NT) to: /content/kg_artifacts/initial_graph.nt
Saved combined ontology+graph to: /content/kg_artifacts/combined_graph.ttl


In [9]:
# Cell 9 — Compute KB statistics and inspect sample triples

from rdflib.term import URIRef

def graph_stats(graph: Graph):
    subjects = set()
    objects = set()
    predicates = set()

    for s, p, o in graph:
        if isinstance(s, URIRef):
            subjects.add(s)
        if isinstance(o, URIRef):
            objects.add(o)
        predicates.add(p)

    entity_nodes = subjects.union(objects)

    return {
        "triples": len(graph),
        "unique_entities": len(entity_nodes),
        "unique_predicates": len(predicates),
    }

ontology_stats = graph_stats(ontology_graph)
kg_stats = graph_stats(kg_graph)
combined_stats = graph_stats(combined_graph)

print("=== Ontology graph statistics ===")
for k, v in ontology_stats.items():
    print(f"{k}: {v}")

print("\n=== Initial RDF graph statistics ===")
for k, v in kg_stats.items():
    print(f"{k}: {v}")

print("\n=== Combined graph statistics ===")
for k, v in combined_stats.items():
    print(f"{k}: {v}")

# Show sample triples
sample_triples = []
for i, (s, p, o) in enumerate(combined_graph):
    sample_triples.append({
        "subject": str(s),
        "predicate": str(p),
        "object": str(o),
    })
    if i >= 14:
        break

sample_triples_df = pd.DataFrame(sample_triples)

print("\n=== Sample triples ===")
display(sample_triples_df)

=== Ontology graph statistics ===
triples: 77
unique_entities: 32
unique_predicates: 5

=== Initial RDF graph statistics ===
triples: 31562
unique_entities: 8292
unique_predicates: 111

=== Combined graph statistics ===
triples: 31639
unique_entities: 8315
unique_predicates: 114

=== Sample triples ===


,subject,predicate,object
0,http://example.org/movie#entity_mrbeast,http://www.w3.org/2000/01/rdf-schema#label,MrBeast
1,http://example.org/movie#film_Q130385353,http://www.w3.org/2000/01/rdf-schema#label,Cherub (film)
2,http://example.org/movie#film_Q130615105,http://example.org/movie#title,Espantaho
3,http://example.org/movie#entity_bronson_pinchot,http://www.w3.org/2000/01/rdf-schema#label,Bronson Pinchot
4,http://example.org/movie#entity_mahmudul_islam...,http://www.w3.org/2000/01/rdf-schema#label,Mahmudul Islam Mithu
5,http://example.org/movie#entity_dai_ikeda,http://www.w3.org/2000/01/rdf-schema#label,Dai Ikeda
6,http://example.org/movie#entity_riya_shibu,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/movie#Person
7,http://example.org/movie#entity_ben_russell,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/movie#Person
8,http://example.org/movie#entity_christian_convery,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/movie#Person
9,http://example.org/movie#entity_seema_biswas,http://www.w3.org/2000/01/rdf-schema#label,Seema Biswas


In [10]:
# Cell 10 — Quick quality checks on the initial KB

# Count how many film nodes were created
film_nodes = set()
for s, p, o in kg_graph.triples((None, RDF.type, EX.Film)):
    film_nodes.add(s)

print("=== Film-level QA ===")
print(f"Film nodes in graph: {len(film_nodes)}")

# Count structured predicate coverage
structured_predicates = [
    EX.directedBy,
    EX.hasCastMember,
    EX.hasGenre,
    EX.hasCountry,
    EX.wonAward,
    EX.producedBy,
    EX.followedBy,
    EX.precededBy,
    EX.mentionsEntity,
]

for predicate in structured_predicates:
    count = sum(1 for _ in kg_graph.triples((None, predicate, None)))
    print(f"{predicate.split('#')[-1]}: {count}")

# Count extracted relation edges
extracted_relation_edges = 0
for predicate_uri, _ in dynamic_predicates:
    extracted_relation_edges += sum(1 for _ in kg_graph.triples((None, predicate_uri, None)))

print(f"\nExtracted relation edges: {extracted_relation_edges}")

# Show one example film subgraph
if film_nodes:
    example_film = next(iter(film_nodes))
    print("\n=== Example film subgraph ===")
    example_rows = []
    for s, p, o in kg_graph.triples((example_film, None, None)):
        example_rows.append({
            "subject": str(s),
            "predicate": str(p),
            "object": str(o),
        })
    display(pd.DataFrame(example_rows).head(20))

=== Film-level QA ===
Film nodes in graph: 1030
directedBy: 666
hasCastMember: 3008
hasGenre: 856
hasCountry: 985
wonAward: 68
producedBy: 271
followedBy: 10
precededBy: 22
mentionsEntity: 5095

Extracted relation edges: 655

=== Example film subgraph ===


,subject,predicate,object
0,http://example.org/movie#film_Q120758628,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/movie#Film
1,http://example.org/movie#film_Q120758628,http://www.w3.org/2000/01/rdf-schema#label,Abigail (2024 film)
2,http://example.org/movie#film_Q120758628,http://example.org/movie#title,Abigail (2024 film)
3,http://example.org/movie#film_Q120758628,http://example.org/movie#directedBy,http://example.org/movie#entity_matt_bettinell...
4,http://example.org/movie#film_Q120758628,http://example.org/movie#directedBy,http://example.org/movie#entity_tyler_gillett
5,http://example.org/movie#film_Q120758628,http://example.org/movie#hasGenre,http://example.org/movie#entity_horror_film
6,http://example.org/movie#film_Q120758628,http://example.org/movie#hasGenre,http://example.org/movie#entity_speculative_fi...
7,http://example.org/movie#film_Q120758628,http://example.org/movie#hasGenre,http://example.org/movie#entity_vampire_film
8,http://example.org/movie#film_Q120758628,http://example.org/movie#hasCountry,http://example.org/movie#entity_united_states
9,http://example.org/movie#film_Q120758628,http://example.org/movie#hasCastMember,http://example.org/movie#entity_alisha_weir


## Notebook 3 summary

This notebook created the **initial private RDF knowledge base** for the movie project by combining:
- structured movie metadata collected in Notebook 1,
- cleaned Wikipedia summaries,
- named entities extracted in Notebook 2,
- and candidate relations extracted in Notebook 2.

### What was added to the graph

The notebook first defined a lightweight ontology tailored to the movie domain.  
This ontology introduced the core classes and predicates needed to represent films and their related entities, including:
- films,
- people,
- organizations,
- places,
- countries,
- genres,
- awards,
- companies,
- and other extracted entities.

Using this ontology, the notebook then constructed the first RDF graph by integrating:
- film-level metadata such as titles and IMDb identifiers,
- country, genre, cast, director, producer, sequel/prequel, and award relations,
- cleaned Wikipedia summaries and page URLs,
- entity mentions extracted from text and linked to films through `mentionsEntity`,
- and additional candidate relation triples derived from relation extraction.

### Main graph quality results

The final graph contains a strong film-centered structure with much broader coverage than in the earlier smaller run:

- **Film nodes:** 1030
- **`directedBy` edges:** 666
- **`hasCastMember` edges:** 3008
- **`hasGenre` edges:** 856
- **`hasCountry` edges:** 985
- **`wonAward` edges:** 68
- **`producedBy` edges:** 271
- **`followedBy` edges:** 10
- **`precededBy` edges:** 22
- **`mentionsEntity` edges:** 5095
- **Extracted relation edges:** 655

These results show that the initial RDF graph is not limited to structured metadata only. It already integrates a large amount of text-derived knowledge through extracted entities and candidate relations.

### Interpretation

The graph produced in this notebook is the first real **project knowledge base**.  
It already captures two complementary sources of information:

1. **Structured movie metadata**, which is relatively reliable and film-centered.
2. **Information extracted from summaries**, which adds coverage and richer connections, but also introduces some noise.

This means the graph is already useful for downstream tasks, while still leaving room for improvement through:
- alignment to Wikidata,
- graph expansion,
- reasoning,
- and cleaning in later notebooks.

### Example graph structure

At the film level, each movie node can now connect to:
- its RDF type and label,
- title and IMDb identifier,
- genre and country information,
- directors, cast, producers, awards, and sequel/prequel relations,
- cleaned summary text and Wikipedia page URL,
- and multiple mentioned entities extracted from text.

This confirms that the graph already combines **symbolic structure** and **text-derived enrichment** in a single RDF representation.

### Main outputs

This notebook produced the following artifacts:
- `/content/kg_artifacts/ontology.ttl`
- `/content/kg_artifacts/initial_graph.ttl`
- `/content/kg_artifacts/initial_graph.nt`
- `/content/kg_artifacts/combined_graph.ttl`

### Why this matters

This notebook is the point where the project moves from raw collected data to a **machine-readable RDF knowledge graph**.

That is important because all later stages depend on this graph:
- Notebook 4 aligns and expands it with Wikidata,
- Notebook 5 performs reasoning and knowledge graph embedding,
- and Notebook 6 uses the final graph for RDF/SPARQL-based question answering.

### Next step

The next notebook focuses on:
1. **entity linking** between the private KB and Wikidata,
2. **predicate alignment** between local predicates and external semantics,
3. and **knowledge graph expansion** using aligned entities.

This initial RDF graph provides the foundation for the remaining project stages:
- alignment,
- expansion,
- reasoning,
- knowledge graph embeddings,
- and RDF/SPARQL-based RAG.